In [68]:
import os
os.getcwd()

'/Users/macass/Desktop/Projet TFE/Génération base de données'

## 1. Importer les librairies

In [69]:
import pandas as pd 
import numpy as np 
import random

# Les données synthétiques sont reproductibles grâce à l'utilisation d'une graine aléatoire fixée.
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

## 2. Génération des données, fonctions et paramètres

#### 2.1 Paramètres

In [70]:
NB_COMPANIES = 10000 
COMPANY_FRAUD_RATE = 0.05 
YEARS = [2020,2021,2022,2023, 2024] 
QUARTERS = ["Q1", "Q2", "Q3","Q4"]

#### 2.2 Codes NACEBEL

In [71]:
NACE_CODES = [ "62010", # programmation informatique 
               "62020", # conseil informatique 
               "46900", # commerce de gros 
               "41201", # construction 
               "56101", # horeca 
               "70220", # conseil 
               "49410", # transport 
               "69201", # comptabilité 
               "47111", # commerce détail 
               "35110" # énergie 
             ]

#### 2.3 TVA belge

In [72]:
vat_numbers = set()

def generate_vat_number():
    while True:
        number = random.randint(100000000, 199999999)
        vat = f"BE{number:09d}"
        if vat not in vat_numbers:
            vat_numbers.add(vat)
            return vat
            # vérif les doublons

#### 2.4 Profil des entreprises

In [73]:
def generate_company_profile(): 
    size = random.choices(
        ["Micro","Petite","Grande","Très Grande"], 
        weights=[60,25,10,5] 
    )[0] 
    if size == "Micro":
        employees = random.randint(1, 10)
        turnover = random.uniform(50000, 700000)
        balance = random.uniform(25000, 350000)
    
    elif size == "Petite":
        employees = random.randint(11, 50)
        turnover = random.uniform(700001, 9000000)
        balance = random.uniform(350001, 4500000)

    elif size == "Grande":
        employees = random.randint(51, 500)
        turnover = random.uniform(9000001, 34000000)
        balance = random.uniform(4500001, 17000000)

    else:
        employees = random.randint(501, 5000)
        turnover = random.uniform(34000001, 300000000)
        balance = random.uniform(17000001, 150000000)

# Une entité d'intérêt public n'est pas automatiquement
# une entreprise "Très Grande".
    
    if size == "Très Grande":
        public_interest = random.random() < 0.30

    else:
        public_interest = random.random() < 0.02
    return { "employees": employees,
             "turnover": turnover, 
             "balance_sheet_total": balance, 
             "public_interest_entity": public_interest, 
             "company_size": size 
           }

#### 2.5 Déclaration TVA

In [74]:
def generate_vat_return(turnover):

    # =========================
    # VENTES
    # =========================

    sales_6_pct = random.uniform(0, 0.10)
    sales_12_pct = random.uniform(0, 0.10)

    # Part des livraisons intracommunautaires
    intra_supplies_pct = random.uniform(0, 0.30)

    sales_6 = turnover * sales_6_pct
    sales_12 = turnover * sales_12_pct

    # Livraisons intracommunautaires
    intra_supplies = turnover * intra_supplies_pct

    # Le reste du chiffre d'affaires correspond
    # aux ventes nationales à 21 %
    sales_21 = max(
        0,
        turnover
        - sales_6
        - sales_12
        - intra_supplies
    )

    # Total des ventes
    sales_total = (
        sales_6
        + sales_12
        + sales_21
        + intra_supplies
    )

    # =========================
    # ACHATS
    # =========================
    
    domestic_purchases_total = turnover * random.uniform(0.40, 0.85)
    
    # Acquisitions intracommunautaires
    intra_acquisitions = (
        domestic_purchases_total * random.uniform(0, 0.25)
    )
    
    purchases_6_pct = random.uniform(0, 0.10)
    purchases_12_pct = random.uniform(0, 0.10)
    
    purchases_6 = domestic_purchases_total * purchases_6_pct
    purchases_12 = domestic_purchases_total * purchases_12_pct
    
    # Achats nationaux à 21 %
    purchases_21 = max(
        0,
        domestic_purchases_total
        - purchases_6
        - purchases_12
    )
    
    # Total des achats, acquisitions intracommunautaires incluses
    purchases_total = (
        purchases_6
        + purchases_12
        + purchases_21
        + intra_acquisitions
    )
    
    
    if intra_supplies > 0:
        nb_supplies = random.randint(1, 200)
    else:
        nb_supplies = 0

    if intra_acquisitions > 0:
        nb_acquisitions = random.randint(1, 150)
    
    else:
        nb_acquisitions = 0
    
    return {
    "sales": sales_total,
    "purchases": purchases_total,

    "sales_6": sales_6,
    "sales_12": sales_12,
    "sales_21": sales_21,

    "intra_community_supplies": intra_supplies,

    "purchases_6": purchases_6,
    "purchases_12": purchases_12,
    "purchases_21": purchases_21,

    "intra_community_acquisitions": intra_acquisitions,

    "nb_intra_supplies": nb_supplies,
    "nb_intra_acquisitions": nb_acquisitions
    }

#### 2.6 Calcul TVA

In [75]:
def compute_vat(data):

    # =========================
    # TVA SUR VENTES
    # =========================

    vat_sales_6 = data["sales_6"] * 0.06
    vat_sales_12 = data["sales_12"] * 0.12
    vat_sales_21 = data["sales_21"] * 0.21

    total_vat_sales = (
        vat_sales_6
        + vat_sales_12
        + vat_sales_21
    )

    # =========================
    # TVA SUR ACHATS
    # =========================

    vat_purchases_6 = data["purchases_6"] * 0.06
    vat_purchases_12 = data["purchases_12"] * 0.12
    vat_purchases_21 = data["purchases_21"] * 0.21

    total_vat_purchases = (
        vat_purchases_6
        + vat_purchases_12
        + vat_purchases_21
    )

    # =========================
    # ACQUISITIONS INTRACOMMUNAUTAIRES
    # =========================

    vat_intra_acquisitions = (
        data["intra_community_acquisitions"] * 0.21
    )

    vat_deductible_intra = (
        data["intra_community_acquisitions"] * 0.21
    )

    # =========================
    # TVA DUE
    # =========================

    vat_due = (
        total_vat_sales
        + vat_intra_acquisitions
        - vat_deductible_intra
        - total_vat_purchases
    )

    # =========================
    # DONNÉES TVA DÉTAILLÉES
    # =========================

    data["vat_sales_6"] = vat_sales_6
    data["vat_sales_12"] = vat_sales_12
    data["vat_sales_21"] = vat_sales_21

    data["vat_purchases_6"] = vat_purchases_6
    data["vat_purchases_12"] = vat_purchases_12
    data["vat_purchases_21"] = vat_purchases_21

    data["total_vat_sales"] = total_vat_sales
    data["total_vat_purchases"] = total_vat_purchases

    data["vat_intra_acquisitions"] = vat_intra_acquisitions
    data["vat_deductible_intra"] = vat_deductible_intra

    data["vat_due"] = vat_due

    return data

#### 2.7 Génération des données

In [76]:
rows = []

for company_id in range(NB_COMPANIES):

    # =========================
    # IDENTITÉ ENTREPRISE
    # =========================

    vat_number = generate_vat_number()

    nace_code = random.choice(NACE_CODES)

    base_profile = generate_company_profile()

    # =========================
    # FRAUDE AU NIVEAU ENTREPRISE
    # =========================

    fraud = random.random() < COMPANY_FRAUD_RATE

    if fraud:
        fraud_type = random.choice([
            "Carousel",
            "FakeInvoices",
            "UnderReporting",
            "ShellCompany"
        ])
    else:
        fraud_type = "None"

    # =========================
    # PROFIL ENTREPRISE
    # =========================

    profile = base_profile.copy()

    initial_age = random.randint(1, 40)

    # =========================
    # SOCIÉTÉ ÉCRAN
    # =========================

    if fraud_type == "ShellCompany":

        profile["employees"] = random.randint(1, 3)

        profile["company_size"] = "Micro"

        profile["public_interest_entity"] = False

        profile["turnover"] = random.uniform(
            50000,
            500000
        )

        profile["balance_sheet_total"] = random.uniform(
            25000,
            300000
        )

    # =========================
    # ANNÉES
    # =========================

    for year in YEARS:

        company_age = (
            initial_age
            + (year - YEARS[0])
        )

        growth = random.uniform(
            0.95,
            1.05
        )

        annual_turnover = (
            profile["turnover"]
            * growth
        )

        annual_turnover *= np.random.lognormal(
            mean=0,
            sigma=0.1
        )

        turnover_effective = annual_turnover

        # =========================
        # SAISONNALITÉ
        # =========================

        seasonality_weights = np.random.dirichlet(
            [4, 4, 4, 4]
        )

        # =========================
        # TRIMESTRES
        # =========================

        for quarter, weight in zip(
            QUARTERS,
            seasonality_weights
        ):

            quarter_turnover = (
                turnover_effective
                * weight
            )

            # =========================
            # DÉCLARATION TVA NORMALE
            # =========================

            decl = generate_vat_return(
                quarter_turnover
            )

            # =========================
            # FRAUDE
            # =========================

            if fraud_type == "Carousel":

                carousel_factor = random.uniform(
                    2.0,
                    5.0
                )

                decl["intra_community_acquisitions"] *= (
                    carousel_factor
                )

                if decl["nb_intra_acquisitions"] > 0:

                    decl["nb_intra_acquisitions"] = int(
                        decl["nb_intra_acquisitions"]
                        * carousel_factor
                    )

            elif fraud_type == "FakeInvoices":

                fake_factor = random.uniform(
                    2.0,
                    3.0
                )

                decl["purchases_21"] *= (
                    fake_factor
                )

            elif fraud_type == "UnderReporting":

                underreporting_factor = random.uniform(
                    0.80,
                    0.95
                )

                decl["sales_6"] *= (
                    underreporting_factor
                )

                decl["sales_12"] *= (
                    underreporting_factor
                )

                decl["sales_21"] *= (
                    underreporting_factor
                )

                decl["intra_community_supplies"] *= (
                    underreporting_factor
                )

            elif fraud_type == "ShellCompany":

                shell_factor = random.uniform(
                    0.60,
                    0.90
                )

                decl["sales_21"] *= (
                    shell_factor
                )

                decl["purchases_21"] *= random.uniform(
                    0.90,
                    1.10
                )

            # =========================
            # RECALCUL DES TOTAUX
            # =========================

            decl["sales"] = (
                decl["sales_6"]
                + decl["sales_12"]
                + decl["sales_21"]
                + decl["intra_community_supplies"]
            )

            decl["purchases"] = (
                decl["purchases_6"]
                + decl["purchases_12"]
                + decl["purchases_21"]
                + decl["intra_community_acquisitions"]
            )

            # =========================
            # CALCUL TVA
            # =========================

            decl = compute_vat(decl)

            # TVA réellement due
            real_vat_due = decl["vat_due"]

            # TVA déclarée
            declared_vat = real_vat_due

            if (
                fraud_type == "UnderReporting"
                and real_vat_due > 0
            ):

                declared_vat = (
                    real_vat_due
                    * random.uniform(0.80, 0.95)
                )

            decl["vat_declared"] = declared_vat

            decl["vat_gap"] = (
                real_vat_due
                - declared_vat
            )

            # =========================
            # FEATURES
            # =========================

            sales = decl["sales"]

            purchases = decl["purchases"]

            real_quarter_turnover = (
                quarter_turnover
            )

            declared_sales = sales

            employees = profile["employees"]

            quarter_turnover_gap = (
                real_quarter_turnover
                - declared_sales
            )

            quarter_turnover_gap_ratio = (
                quarter_turnover_gap
                / quarter_turnover
                if quarter_turnover > 0
                else 0
            )

            # =========================
            # CRÉATION DE LA LIGNE
            # =========================

            row = {
                "vat_number": vat_number,

                "year": year,

                "quarter": quarter,

                "nacebel_code": nace_code,

                "company_size": profile[
                    "company_size"
                ],

                "employees": profile[
                    "employees"
                ],

                "turnover": annual_turnover,

                "quarter_turnover": (
                    real_quarter_turnover
                ),

                "declared_sales": (
                    declared_sales
                ),

                "balance_sheet_total": profile[
                    "balance_sheet_total"
                ],

                "company_age": company_age,

                "public_interest_entity": profile[
                    "public_interest_entity"
                ],

                "quarter_turnover_gap": (
                    quarter_turnover_gap
                ),

                "quarter_turnover_gap_ratio": (
                    quarter_turnover_gap_ratio
                ),

                "fraud": int(fraud),

                "fraud_type": fraud_type
            }

            # Ajout des données TVA
            row.update(decl)

            # =========================
            # VARIABLES FINANCIÈRES
            # =========================

            row["profit_margin"] = (
                (sales - purchases) / sales
                if sales > 0
                else 0
            )

            row["input_output_ratio"] = (
                purchases / sales
                if sales > 0
                else 0
            )

            row["turnover_per_employee"] = (
                annual_turnover / employees
                if employees > 0
                else 0
            )

            rows.append(row)

## 3. Construction du Dataframe

In [77]:
df = pd.DataFrame(rows) 

## 4. Contrôle qualité

#### 4.1 Vérification de la qualité des données

In [78]:
print("Dimensions du DataFrame :", df.shape)

# Vérification des valeurs positives
assert df["sales"].ge(0).all()
assert df["purchases"].ge(0).all()
assert df["employees"].gt(0).all()
assert df["turnover"].gt(0).all()
assert df["intra_community_supplies"].ge(0).all()
assert df["intra_community_acquisitions"].ge(0).all()
assert df["nb_intra_supplies"].ge(0).all()
assert df["nb_intra_acquisitions"].ge(0).all()
assert df["turnover_per_employee"].ge(0).all()

# Vérification des valeurs manquantes
print("\nValeurs manquantes :")
print(df.isna().sum().sum())

# Vérification des entreprises
print("\nNombre d'entreprises uniques :")
print(df["vat_number"].nunique())

# Vérification du nombre de lignes
print("\nNombre de lignes :", len(df))

# Chaque entreprise doit avoir 20 déclarations
company_rows = df.groupby("vat_number").size()

assert (company_rows == 20).all()

# Vérification des années et trimestres
assert set(df["year"].unique()) == set(YEARS)
assert set(df["quarter"].unique()) == set(QUARTERS)

# Vérification du taux de fraude
print("\nTaux de fraude :")
print(df["fraud"].mean())

# Vérification des types de fraude
print("\nTypes de fraude :")
print(df["fraud_type"].value_counts())

# Cohérence entre fraud et fraud_type
assert (
    ((df["fraud"] == 0) & (df["fraud_type"] == "None")) |
    ((df["fraud"] == 1) & (df["fraud_type"] != "None"))
).all()

# Vérification de la cohérence entre CA annuel et CA trimestriel
annual_quarter_turnover = (
    df.groupby(["vat_number", "year"])["quarter_turnover"].sum()
)

annual_turnover = (
    df.groupby(["vat_number", "year"])["turnover"].first()
)

assert np.isclose(
    annual_quarter_turnover,
    annual_turnover
).all()

Dimensions du DataFrame : (200000, 44)

Valeurs manquantes :
0

Nombre d'entreprises uniques :
10000

Nombre de lignes : 200000

Taux de fraude :
0.051

Types de fraude :
fraud_type
None              189800
FakeInvoices        2780
ShellCompany        2520
UnderReporting      2460
Carousel            2440
Name: count, dtype: int64


#### 4.2 Cohérence des ventes

In [79]:
# Le chiffre d'affaires déclaré doit correspondre
# à la somme des composantes de ventes
sales_check = np.isclose(
    df["sales"],
    df["sales_6"]
    + df["sales_12"]
    + df["sales_21"]
    + df["intra_community_supplies"]
)

print(
    "\nCohérence ventes :",
    sales_check.mean() * 100,
    "%"
)

assert sales_check.all()


Cohérence ventes : 100.0 %


#### 4.3 Cohérence des achats

In [80]:
# Les achats doivent correspondre à la somme
# des différentes catégories
purchases_check = np.isclose(
    df["purchases"],
    df["purchases_6"]
    + df["purchases_12"]
    + df["purchases_21"]
    + df["intra_community_acquisitions"]
)

print(
    "Cohérence achats :",
    purchases_check.mean() * 100,
    "%"
)

assert purchases_check.all()

Cohérence achats : 100.0 %


#### 4.4 Cohérence TVA

In [81]:
# Cohérence TVA sur ventes
assert np.isclose(
    df["vat_sales_6"],
    df["sales_6"] * 0.06
).all()

assert np.isclose(
    df["vat_sales_12"],
    df["sales_12"] * 0.12
).all()

assert np.isclose(
    df["vat_sales_21"],
    df["sales_21"] * 0.21
).all()

# Cohérence TVA sur achats
assert np.isclose(
    df["vat_purchases_6"],
    df["purchases_6"] * 0.06
).all()

assert np.isclose(
    df["vat_purchases_12"],
    df["purchases_12"] * 0.12
).all()

assert np.isclose(
    df["vat_purchases_21"],
    df["purchases_21"] * 0.21
).all()

# Cohérence TVA sur acquisitions intracommunautaires
assert np.isclose(
    df["vat_intra_acquisitions"],
    df["intra_community_acquisitions"] * 0.21
).all()

assert np.isclose(
    df["vat_deductible_intra"],
    df["intra_community_acquisitions"] * 0.21
).all()

# Cohérence du calcul de la TVA due
vat_due_check = (
    df["total_vat_sales"]
    + df["vat_intra_acquisitions"]
    - df["vat_deductible_intra"]
    - df["total_vat_purchases"]
)

assert np.isclose(
    df["vat_due"],
    vat_due_check
).all()

# Cohérence de l'écart de TVA
assert np.isclose(
    df["vat_gap"],
    df["vat_due"] - df["vat_declared"]
).all()

In [82]:
# contrôl du taux de fraude
company_fraud = (
    df.groupby("vat_number")["fraud"]
    .max()
)

print(company_fraud.mean())

0.051


## 5. Export

In [83]:
df.to_csv( "simulation_vat_fraud.csv", index=False, sep="," ) 

In [84]:
print(df.head()) 
print() 
print("Nombre de lignes :", len(df)) 
print("CSV généré : simulation_vat_fraud.csv")

    vat_number  year quarter nacebel_code company_size  employees  \
0  BE185822412  2020      Q1        62020        Micro          5   
1  BE185822412  2020      Q2        62020        Micro          5   
2  BE185822412  2020      Q3        62020        Micro          5   
3  BE185822412  2020      Q4        62020        Micro          5   
4  BE185822412  2021      Q1        62020        Micro          5   

        turnover  quarter_turnover  declared_sales  balance_sheet_total  ...  \
0  210751.912393      51785.450606    51785.450606         70349.826771  ...   
1  210751.912393      29219.767899    29219.767899         70349.826771  ...   
2  210751.912393      65515.148831    65515.148831         70349.826771  ...   
3  210751.912393      64231.545056    64231.545056         70349.826771  ...   
4  236156.799606      52042.277852    52042.277852         70349.826771  ...   

   total_vat_sales  total_vat_purchases  vat_intra_acquisitions  \
0      9820.013131          6536.1052

In [85]:
import os
print(os.getcwd())

/Users/macass/Desktop/Projet TFE/Génération base de données


In [86]:
from IPython.display import FileLink

FileLink("simulation_vat_fraud.csv")

/Users/macass/Desktop/Projet TFE/Génération base de données/simulation_vat_fraud.csv

In [87]:
print(df['fraud'].value_counts())

fraud
0    189800
1     10200
Name: count, dtype: int64


In [88]:
print(df['fraud'].value_counts(normalize=True))

fraud
0    0.949
1    0.051
Name: proportion, dtype: float64


In [89]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 44 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   vat_number                    200000 non-null  object 
 1   year                          200000 non-null  int64  
 2   quarter                       200000 non-null  object 
 3   nacebel_code                  200000 non-null  object 
 4   company_size                  200000 non-null  object 
 5   employees                     200000 non-null  int64  
 6   turnover                      200000 non-null  float64
 7   quarter_turnover              200000 non-null  float64
 8   declared_sales                200000 non-null  float64
 9   balance_sheet_total           200000 non-null  float64
 10  company_age                   200000 non-null  int64  
 11  public_interest_entity        200000 non-null  bool   
 12  quarter_turnover_gap          200000 non-nul

In [90]:
df.shape

(200000, 44)

In [91]:
print(df["fraud"].value_counts())

fraud
0    189800
1     10200
Name: count, dtype: int64


In [92]:
print(df["fraud"].value_counts(normalize=True))

fraud
0    0.949
1    0.051
Name: proportion, dtype: float64


In [93]:
df.groupby("fraud")["turnover"].mean()

fraud
0    1.183817e+07
1    9.943339e+06
Name: turnover, dtype: float64

In [94]:
df.groupby("fraud")["employees"].mean()

fraud
0    171.066596
1    159.454902
Name: employees, dtype: float64

In [95]:
print("=== DIMENSIONS ===")
print(df.shape)

print("\n=== FRAUDE PAR LIGNE ===")
print(df["fraud"].value_counts())

print("\n=== POURCENTAGE FRAUDE PAR LIGNE ===")
print(df["fraud"].value_counts(normalize=True).mul(100).round(2))

print("\n=== TYPES DE FRAUDE ===")
print(df["fraud_type"].value_counts())

print("\n=== ENTREPRISES UNIQUES ===")
print(df["vat_number"].nunique())

print("\n=== FRAUDE AU NIVEAU ENTREPRISE ===")
company_fraud = df.groupby("vat_number")["fraud"].max()
print(company_fraud.value_counts())

print("\n=== TAUX DE FRAUDE ENTREPRISE ===")
print(company_fraud.mean())

=== DIMENSIONS ===
(200000, 44)

=== FRAUDE PAR LIGNE ===
fraud
0    189800
1     10200
Name: count, dtype: int64

=== POURCENTAGE FRAUDE PAR LIGNE ===
fraud
0    94.9
1     5.1
Name: proportion, dtype: float64

=== TYPES DE FRAUDE ===
fraud_type
None              189800
FakeInvoices        2780
ShellCompany        2520
UnderReporting      2460
Carousel            2440
Name: count, dtype: int64

=== ENTREPRISES UNIQUES ===
10000

=== FRAUDE AU NIVEAU ENTREPRISE ===
fraud
0    9490
1     510
Name: count, dtype: int64

=== TAUX DE FRAUDE ENTREPRISE ===
0.051
